# v10 — NVFP4 KV cache — the gate (T4-EMULATED, Colab T4)

Forks v9 (FP8) changing ONE variable: KV **storage format** → NVFP4 (packed 4-bit E2M1 nibble + one
E4M3 micro-scale per 16 elems = 0.5625 B/elem). Score-stationary loop / M-packing / split-KV / merge
byte-identical → clean byte-only A/B vs v9 *and* v8.7.

⚠️ **This is the T4-EMULATED path: correctness + capacity + accuracy ONLY.** Software FP4 unpack is
*more* ALU than v9's E4M3 (which already power-capped the T4 clock 1590→1350), so **µs/tok here is NOT
a valid latency number** — trust only same-session clock-matched ratios, and the real latency/regime/
sm_103-exp deliverables run on a root B300 (`docs/v10-kickoff.md` §5–6). Decode uses NO FP4 tensor
cores (M=G<64 < Blackwell's M≥64 gate); native FP4 compute is v11.

## 0. Dependencies + GPU (venv-safe)

In [1]:
import os, sys, subprocess

def pip(*pkgs, extra=()):
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *pkgs, *extra], check=True)

# 1) Physical GPU on this runtime? (Colab defaults to CPU; pick a GPU explicitly.)
try:
    has_gpu = subprocess.run(['nvidia-smi'], capture_output=True).returncode == 0
except FileNotFoundError:
    has_gpu = False
if not has_gpu:
    raise SystemExit(
        'No GPU on this Colab runtime. FIX: Runtime > Change runtime type > T4 GPU > Save, '
        'then Runtime > Restart session, then re-run from the top. (This kernel needs a Turing T4.)')

# 2) Install deps (incl. numpy) BEFORE importing torch, so torch's numpy bridge initializes.
pip('ninja', 'pytest', 'numpy')

# 3) torch present AND CUDA-enabled? A CPU-only wheel raises "not compiled with CUDA" on any kernel.
try:
    import torch
    cuda_ok = torch.cuda.is_available()
except ImportError:
    torch, cuda_ok = None, False

if not cuda_ok:
    pip('torch', extra=('--index-url', 'https://download.pytorch.org/whl/cu124'))
    raise SystemExit(
        'A GPU is present but torch was a CPU-only build -- installed the CUDA build. NOW: '
        'restart the kernel/session, then re-run this cell.')

# vast.ai/venv: !-cells spawn a bare shell without the venv on PATH -> `python` not found.
os.environ['PATH'] = os.path.dirname(sys.executable) + os.pathsep + os.environ.get('PATH', '')

# torch.float8_e4m3fn must exist (>=2.1) — v9 stores the KV cache as E4M3 bytes.
assert hasattr(torch, 'float8_e4m3fn'), 'this torch lacks float8_e4m3fn; upgrade torch (>=2.1)'
print('torch', torch.__version__, '| cuda', torch.version.cuda, '| cap', torch.cuda.get_device_capability())
!nvidia-smi --query-gpu=name,compute_cap --format=csv

torch 2.11.0+cu128 | cuda 12.8 | cap (7, 5)
name, compute_cap
Tesla T4, 7.5


## 1. Get the repo

In [2]:
REPO_URL = 'https://github.com/gkienpham-cmd/flashattention-cuda.git'  # public; plain clone works
import os, sys, subprocess
if os.path.basename(os.getcwd()) != 'flashattention-cuda':
    if not os.path.isdir('flashattention-cuda'):
        subprocess.run(['git', 'clone', REPO_URL], check=True)
    os.chdir('flashattention-cuda')
subprocess.run(['git', 'pull', 'origin', 'main'])
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())
print('cwd', os.getcwd())

cwd /content/flashattention-cuda


## 2. Roofline — NVFP4 ~3.55× lower HBM floor than FP16 (still BLIND, recorded BEFORE coding)

In [3]:
from roofline.archs import get_arch
from roofline.model import estimate
# Run box is the T4 (sm_75); the paper's prediction is on B300 (sm_103). Show NVFP4 vs FP8 vs FP16
# decode AI = 2G/b (b: fp16=2, fp8=1, nvfp4=0.5625). At G=8: fp16=8.0, fp8=16.0, nvfp4=28.4.
for sm in ('sm_75', 'sm_103'):
    arch = get_arch(sm)
    print(f'\n=== {arch.name} ({sm}) | HBM {arch.hbm_bw_gbps} GB/s ===')
    print(f"{'G':>3} | {'AI fp16':>8} | {'AI fp8':>8} | {'AI nvfp4':>9} | {'limiter':>7} | {'floor cut vs fp16':>17}")
    for G in (1, 2, 4, 8, 16, 32):
        e16 = estimate(arch, B=8, H=8, N_q=1, N_k=8192, d=128, precision='fp16',  G=G)
        e8  = estimate(arch, B=8, H=8, N_q=1, N_k=8192, d=128, precision='fp8',   G=G)
        e4  = estimate(arch, B=8, H=8, N_q=1, N_k=8192, d=128, precision='nvfp4', G=G)
        print(f'{G:>3} | {e16.arithmetic_intensity:8.1f} | {e8.arithmetic_intensity:8.1f} | '
              f'{e4.arithmetic_intensity:9.1f} | {e4.limiter.upper():>7} | {e16.t_hbm/e4.t_hbm:16.2f}x')
print('\nNVFP4 cuts the HBM floor ~3.55x vs FP16 (0.5625 vs 2 B/elem); limiter STAYS HBM, far below')
print('every ridge. The model is BLIND to the per-CTA wall (v9 Task 1: ~28% HBM cap), the dequant tax,')
print('and the L2. PREDICTION: capacity ~3.55x certain; accuracy delta = the headline; latency win only')
print('fragile-L2-resident (shrinks with G, likely flips negative under flush) OR a B300 bandwidth regime')
print('IFF one exists. COUNTER (the prize): %HBM climbs past the ~126 MB B300 L2 -> bandwidth-bound on sm_103.')


=== Tesla T4 (sm_75) | HBM 320.0 GB/s ===
  G |  AI fp16 |   AI fp8 |  AI nvfp4 | limiter | floor cut vs fp16
  1 |      1.0 |      2.0 |       3.6 |     HBM |             3.56x
  2 |      2.0 |      4.0 |       7.1 |     HBM |             3.56x
  4 |      4.0 |      8.0 |      14.2 |     HBM |             3.56x
  8 |      8.0 |     16.0 |      28.4 |     HBM |             3.56x
 16 |     16.0 |     31.9 |      56.8 |     HBM |             3.56x
 32 |     31.9 |     63.8 |     113.3 |     HBM |             3.56x

=== NVIDIA B300 (Blackwell Ultra, GB300) (sm_103) | HBM 8000.0 GB/s ===
  G |  AI fp16 |   AI fp8 |  AI nvfp4 | limiter | floor cut vs fp16
  1 |      1.0 |      2.0 |       3.6 |     HBM |             3.56x
  2 |      2.0 |      4.0 |       7.1 |     HBM |             3.56x
  4 |      4.0 |      8.0 |      14.2 |     HBM |             3.56x
  8 |      8.0 |     16.0 |      28.4 |     HBM |             3.56x
 16 |     16.0 |     31.9 |      56.8 |     HBM |             3.56x


## 3. Build v10_nvfp4 (JIT) — watch ptxas (emulated FP4 unpack is software ALU on sm_75)

In [4]:
import glob, os, shutil
for d in glob.glob(os.path.expanduser('~/.cache/torch_extensions/*/fa_v10_nvfp4')):
    if not glob.glob(os.path.join(d, '*.so')):
        shutil.rmtree(d, ignore_errors=True); print('cleaned stale build:', d)
from bindings.load import build_kernel
# The NVFP4 nibble unpack is pure integer/ALU (portable); the per-16 micro-scale uses E4M3
# (__nv_cvt_fp8_to_halfraw, software-emulated on sm_75 like v9). If ptxas rejects E4M3, the same
# 1-line int8 fallback as v9 applies (kernels/v10_nvfp4/nvfp4_attention.cu dequant_e4m3).
nvfp4 = build_kernel('v10_nvfp4'); print('built v10 (NVFP4 KV):', nvfp4)

built v10 (NVFP4 KV): <module 'fa_v10_nvfp4' from '/root/.cache/torch_extensions/py312_cu128/fa_v10_nvfp4/fa_v10_nvfp4.so'>


## 4. Correctness gate — v10_nvfp4 + v9/v8.7 regression (Gate 1 of 2)

In [5]:
!python -m pytest tests/test_correctness.py -k "v10_nvfp4 or v9_fp8 or v8_gqa_ss" -q

........................................................................ [ 49%]
........................................................................ [ 98%]
..                                                                       [100%]
146 passed, 316 deselected in 216.97s (0:03:36)


## 5. Accuracy — NVFP4 RMSE vs fp16 AND vs FP8 (is FP4 enough, or is FP8 the floor?)

In [6]:
# The accuracy deliverable. Three numbers per shape:
#   (1) kernel vs apples-to-apples oracle (SDPA on the SAME dequantized NVFP4 bytes) -> FP16 band (gated 5e-2);
#   (2) kernel vs the ORIGINAL fp16 KV -> the NVFP4 QUANTIZATION RMSE (the real number);
#   (3) FP8 quantization RMSE vs fp16 KV -> the comparison: is FP4 acceptable, or is FP8 the accuracy floor?
import torch
from fa_kernels import nvfp4_attention
from fa_kernels.paged import (build_paged_kv_nvfp4, quantize_nvfp4, dequantize_nvfp4,
                              quantize_fp8_e4m3, dequantize_fp8_e4m3)
from fa_kernels.reference import sdpa_reference_gqa, sdpa_reference_gqa_nvfp4

def rmse(a, b):
    return (a - b).pow(2).mean().sqrt().item()

print(f"{'shape':>16} | {'vs NVFP4 oracle':>15} | {'NVFP4 vs fp16':>14} | {'FP8 vs fp16':>12} | verdict")
for d in (64, 128):
    for G in (1, 8):
        for seed in (9, 17):   # vary the seed (v9 was single-seed)
            torch.manual_seed(seed)
            B, H_kv, N_k = 1, 2, 8192
            H_q = G * H_kv; ps = 128
            q = torch.randn(B, H_q,  1,   d, device='cuda')
            k = torch.randn(B, H_kv, N_k, d, device='cuda')
            v = torch.randn(B, H_kv, N_k, d, device='cuda')
            kp, km, vp, vm, bt, nk, sk, sv = build_paged_kv_nvfp4(k, v, ps, seed=seed)
            out = nvfp4_attention(q, kp, km, vp, vm, bt, ps, nk, sk, sv, causal=False, q_offset=0)
            r_or = rmse(out, sdpa_reference_gqa_nvfp4(q, k, v, causal=False))   # apples-to-apples
            r4   = rmse(out, sdpa_reference_gqa(q, k, v, causal=False))         # NVFP4 quant error
            # FP8 quant RMSE on the same data (KV dequantized through E4M3) for the comparison.
            kb, sk8 = quantize_fp8_e4m3(k); vb, sv8 = quantize_fp8_e4m3(v)
            k8 = dequantize_fp8_e4m3(kb, sk8).to(k.dtype); v8 = dequantize_fp8_e4m3(vb, sv8).to(v.dtype)
            r8 = rmse(sdpa_reference_gqa(q, k8, v8, causal=False), sdpa_reference_gqa(q, k, v, causal=False))
            verdict = 'FP4~FP8' if r4 < 2*r8 else 'FP8 floor?'
            print(f"{f'{H_q}x1x{d}/{N_k} G{G} s{seed}':>16} | {r_or:15.2e} | {r4:14.2e} | {r8:12.2e} | {verdict}")
print('\n(1) sits in the FP16 band; (2) is the NVFP4 quant RMSE (the deliverable); (3) FP8 for comparison.')
print('If NVFP4 RMSE >> FP8, the honest result is "FP8 is the accuracy floor; FP4 buys capacity at cost X".')

           shape | vs NVFP4 oracle |  NVFP4 vs fp16 |  FP8 vs fp16 | verdict
2x1x64/8192 G1 s9 |        6.86e-06 |       2.37e-03 |     6.03e-04 | FP8 floor?
2x1x64/8192 G1 s17 |        6.37e-06 |       2.07e-03 |     7.25e-04 | FP8 floor?
16x1x64/8192 G8 s9 |        9.59e-06 |       2.67e-03 |     7.26e-04 | FP8 floor?
16x1x64/8192 G8 s17 |        6.92e-06 |       2.39e-03 |     6.83e-04 | FP8 floor?
2x1x128/8192 G1 s9 |        6.80e-06 |       2.41e-03 |     6.34e-04 | FP8 floor?
2x1x128/8192 G1 s17 |        6.35e-06 |       2.43e-03 |     5.88e-04 | FP8 floor?
16x1x128/8192 G8 s9 |        7.09e-06 |       2.57e-03 |     7.00e-04 | FP8 floor?
16x1x128/8192 G8 s17 |        7.30e-06 |       2.44e-03 |     6.77e-04 | FP8 floor?

(1) sits in the FP16 band; (2) is the NVFP4 quant RMSE (the deliverable); (3) FP8 for comparison.
If NVFP4 RMSE >> FP8, the honest result is "FP8 is the accuracy floor; FP4 buys capacity at cost X".


## 6. Capacity — the durable headline (~3.55× vs FP16, ~1.78× vs FP8); MEASURE it

In [7]:
# v9 ASSERTED 2x capacity but never measured it. Close that gap: report the actual KV-pool footprint
# (bytes) for FP16 vs FP8 vs NVFP4 on a fixed KV, by construction AND via the pool tensors themselves.
import torch
from fa_kernels.paged import build_paged_kv, build_paged_kv_fp8, build_paged_kv_nvfp4
B, H_kv, N_k, d, ps = 1, 8, 65536, 128, 256
k = torch.randn(B, H_kv, N_k, d, device='cuda', dtype=torch.float16)
v = torch.randn(B, H_kv, N_k, d, device='cuda', dtype=torch.float16)
def pool_bytes(*tensors):
    return sum(t.numel() * t.element_size() for t in tensors)
k16, v16, _, _ = build_paged_kv(k, v, ps)
k8, v8, _, _, _, _ = build_paged_kv_fp8(k, v, ps)
kp, kmi, vp, vmi, _, _, _, _ = build_paged_kv_nvfp4(k, v, ps)
b16 = pool_bytes(k16, v16)
b8  = pool_bytes(k8, v8)
b4  = pool_bytes(kp, kmi, vp, vmi)   # packed nibbles + micro-scales (count BOTH)
print(f'KV pool footprint for [{B},{H_kv},{N_k},{d}] (page_size {ps}):')
print(f'  FP16  : {b16/1e6:8.2f} MB  ({b16/(B*H_kv*N_k*d):.4f} B/elem)')
print(f'  FP8   : {b8/1e6:8.2f} MB  ({b8/(B*H_kv*N_k*d):.4f} B/elem)  -> {b16/b8:.2f}x vs FP16')
print(f'  NVFP4 : {b4/1e6:8.2f} MB  ({b4/(B*H_kv*N_k*d):.4f} B/elem)  -> {b16/b4:.2f}x vs FP16, {b8/b4:.2f}x vs FP8')
print('\nExpect ~0.5625 B/elem and ~3.55x vs FP16, ~1.78x vs FP8 — the capacity headline, now MEASURED.')

KV pool footprint for [1,8,65536,128] (page_size 256):
  FP16  :   268.44 MB  (4.0000 B/elem)
  FP8   :   134.22 MB  (2.0000 B/elem)  -> 2.00x vs FP16
  NVFP4 :    75.50 MB  (1.1250 B/elem)  -> 3.56x vs FP16, 1.78x vs FP8

Expect ~0.5625 B/elem and ~3.55x vs FP16, ~1.78x vs FP8 — the capacity headline, now MEASURED.


## 7. THE A/B — v10 vs v9 vs v8.7 (byte-isolated), G-sweep  ⚠️ T4 latency NOT valid

In [8]:
# Same score-stationary loop in all three; ONLY KV storage differs (FP16 -> FP8 -> NVFP4). On T4 the
# emulated FP4 unpack confounds us/tok (software ALU) -> read this for the byte-isolation SHAPE and
# %HBM trend, NOT absolute latency (that's the root-B300 deliverable). 'vs naive' = ratio vs v8.7 FP16.
print('=== v8.7: v8_gqa_ss (FP16 KV) ===')
!python -m bench.harness --backend v8_gqa_ss --decode --seq 8192 --heads 32 --gqa-group 1 2 4 8 16 32
print('\n=== v9: v9_fp8 (FP8 KV) ===')
!python -m bench.harness --backend v9_fp8 --decode --seq 8192 --heads 32 --gqa-group 1 2 4 8 16 32
print('\n=== v10: v10_nvfp4 (NVFP4 KV) ===')
!python -m bench.harness --backend v10_nvfp4 --decode --seq 8192 --heads 32 --gqa-group 1 2 4 8 16 32

=== v8.7: v8_gqa_ss (FP16 KV) ===
# device: Tesla T4 (sm_75)  clock~1365/1590MHz  backend=v8_gqa_ss  precision=fp32  causal=False  decode=True
#       shape(q x kv) |    ours p50/max ms |   us/tok |   %HBM |  vs sdpa | vs naive | roofline
ninja: no work to do.
[1/3] c++ -MMD -MF binding.o.d -DTORCH_EXTENSION_NAME=fa_v7_paged -DTORCH_API_INCLUDE_EXTENSION_H -isystem /usr/local/lib/python3.12/dist-packages/torch/include -isystem /usr/local/lib/python3.12/dist-packages/torch/include/torch/csrc/api/include -isystem /usr/local/cuda/include -isystem /usr/include/python3.12 -fPIC -std=c++17 -c /content/flashattention-cuda/kernels/v7_paged/binding.cpp -o binding.o 
[2/3] /usr/local/cuda/bin/nvcc -MD -MF paged_attention.cuda.o.d -DTORCH_EXTENSION_NAME=fa_v7_paged -DTORCH_API_INCLUDE_EXTENSION_H -isystem /usr/local/lib/python3.12/dist-packages/torch/include -isystem /usr/local/lib/python3.12/dist-packages/torch/include/torch/csrc/api/include -isystem /usr/local/cuda/include -isystem /usr/include

## 8. Reclaim-at-batch (G=8) — does cutting KV bytes move µs/tok at B≥8?  ⚠️ T4 latency NOT valid

In [9]:
print('=== v8.7: v8_gqa_ss (FP16 KV) ===')
!python -m bench.harness --backend v8_gqa_ss --decode --seq 8192 --heads 8 --gqa-group 8 --batch-sweep 1 8 16 32 64
print('\n=== v9: v9_fp8 (FP8 KV) ===')
!python -m bench.harness --backend v9_fp8 --decode --seq 8192 --heads 8 --gqa-group 8 --batch-sweep 1 8 16 32 64
print('\n=== v10: v10_nvfp4 (NVFP4 KV) ===')
!python -m bench.harness --backend v10_nvfp4 --decode --seq 8192 --heads 8 --gqa-group 8 --batch-sweep 1 8 16 32 64

=== v8.7: v8_gqa_ss (FP16 KV) ===
# device: Tesla T4 (sm_75)  clock~585/1590MHz  backend=v8_gqa_ss  precision=fp32  causal=False  decode=True
#       shape(q x kv) |    ours p50/max ms |   us/tok |   %HBM |  vs sdpa | vs naive | roofline
ninja: no work to do.
ninja: no work to do.
     1x8x1x64/8192 G8 |   0.174/  0.276 |    21.70 |   3.8% |   15.85x |    4.54x | HBM (~0.01ms)
    1x8x1x128/8192 G8 |   0.213/  0.242 |    26.62 |   6.2% |    8.72x |    4.62x | HBM (~0.01ms)
     8x8x1x64/8192 G8 |   0.515/  0.534 |     8.05 |  10.2% |   10.24x |    9.40x | HBM (~0.05ms)
    8x8x1x128/8192 G8 |   0.966/  1.022 |    15.09 |  10.9% |    8.21x |    8.28x | HBM (~0.10ms)
    16x8x1x64/8192 G8 |   1.019/  1.210 |     7.96 |  10.3% |   10.39x |    9.48x | HBM (~0.10ms)
   16x8x1x128/8192 G8 |   1.968/  2.089 |    15.38 |  10.7% |    8.03x |    8.13x | HBM (~0.21ms)
    32x8x1x64/8192 G8 |   2.057/  2.155 |     8.03 |  10.2% |    9.83x |    9.35x | HBM (~0.21ms)
   32x8x1x128/8192 G8 |   4.020/